## Import Stuff

In [1]:
%run "/Users/audreyburggraf/Desktop/QUEEN'S/THESIS RESEARCH/PLOTTING C29 989/constants.py"

/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (
/opt/anaconda3/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3432: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/anaconda3/lib/python3.9/site-packages/numpy/core/_methods.py:190: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/opt/anaconda3/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3432: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/anaconda3/lib/python3.9/site-packa

In [2]:
%run "/Users/audreyburggraf/Desktop/QUEEN'S/THESIS RESEARCH/PLOTTING C29 989/FUNCTIONS/load_functions"

fortran mie routines unavailable


/opt/anaconda3/lib/python3.9/site-packages/dsharp_opac/dsharp_opac.py:47: UserWarning: could not import compiled mie code - mie calculation will be slow
  warnings.warn(


In [3]:
%matplotlib inline

In [4]:
bands        = ["Band 4", "Band 4 nterms2", "Band 4 nterms2 smooth", "Band 4 nterms2 smooth B6", "Band 4 nterms2 smooth B6 B7", "Band 5", "Band 5 robust -1", "Band 5 robust -2", "Band 6", "Band 6 smooth", "Band 6 smooth B7", "Band 7 nterms2", "Band 7 nterms2 smooth", "Band 7 nterms2 smooth B6"]
bands_naming = ["Band 4", "Band4_nterms2",  "Band4_nterms2_smooth",  "Band4_nterms2_smooth_B6",  'Band4_nterms2_smooth_B6_B7',  'Band5',  "Band5_robust_minus1", "Band5_robust_minus2", "Band6",  'Band6_smooth',  'Band6_smooth_B7',  "Band7_nterms2", 'Band7_nterms2_smooth',    'Band7_nterms2_smooth_B6']

lambda_bands_cm = mm_to_cm([lambda_mm[b] for b in bands])

## Set $\lambda$ and $a_{max}$ we want to test

In [5]:
# Set up wavelength array
logwave_vals = np.linspace(0.1, 4, 10000)

lambda_dist_micron = 10**logwave_vals

lambda_dist_cm = micron_to_cm(lambda_dist_micron)

In [6]:
# a_max values for this plot
a_max_test_micron = [1, 100]
a_max_test_cm = micron_to_cm(a_max_test_micron)

In [7]:
# # a_max_dist_micron =  np.logspace(np.log10(1e0), np.log10(1e4), N_grains)

# a_max_dist_micron = np.arange(50, 501)

# N_grains = len(a_max_dist_micron)

# a_max_dist_cm = micron_to_cm(a_max_dist_micron)

In [8]:
f_values = [1, 0.75, 0.5, 0.25, 0.3, 0.1, 0.01]

# f_values = [0.5]

In [9]:
af_min_bound_micron = {
    1: 50,
    0.75: 50,
    0.5: 30,
    0.25: 5,
    0.3: 10,
    0.1: 10,
    0.01: 10
}

af_max_bound_micron = {
    1: 501,
    0.75: 501,
    0.5: 1000,
    0.25: 1000,
    0.3: 1000,
    0.1: 1000,
    0.01: 1000
}

N_grains_f = {
    1: (501 - 50) + 1,
    0.75: (501 - 50) + 1,
    0.5: 500,
    0.25: 500,
    0.3: 500,
    0.1: 500,
    0.01: 500
}


In [10]:
# # Define regimes
# compact_f = [1, 0.75, 0.5]
# fluffy_f  = [0.3, 0.25, 0.1, 0.01]

# af_min_bound_micron = {}
# af_max_bound_micron = {}
# N_grains_f = {}

# # Compact grains
# for f in compact_f:
#     af_min_bound_micron[f] = 50
#     af_max_bound_micron[f] = 501
#     N_grains_f[f] = (501 - 50) + 1

# # Fluffy grains
# for f in fluffy_f:
#     af_min_bound_micron[f] = 10
#     af_max_bound_micron[f] = 1000
#     N_grains_f[f] = 500


In [11]:
for f in f_values:
    
    print(rf"Now working on f = {f}")
    
#     a_max_f_dist_cm = micron_to_cm(1/f * np.arange(af_min_bound_micron[f]/f, af_max_bound_micron[f]/f))


    a_max_f_dist_cm = micron_to_cm((1/f) * np.linspace(af_min_bound_micron[f]/f,
                            af_max_bound_micron[f]/f,
                            N_grains_f[f]))
    
    
    # Run DSHARP for this f
    P, omega, P_times_omega = run_DSHARP(f, a_max_test_cm, a_max_f_dist_cm, lambda_bands_cm, lambda_dist_cm)
    
    # Prepare data dictionary
    data = {
        "a_max_f_micron": a_max_f_dist_cm * 1e4
    }
    
    # Add per-band columns
    for i, b in enumerate(bands_naming):  # bands = [4, 5, 6, ...]
        data[f"P_{b}"] = P[:, i]
        data[f"omega_{b}"] = omega[:, i]
        data[f"P_times_omega_{b}"] = P_times_omega[:, i]
    
    # Convert to DataFrame
    df = pd.DataFrame(data)
    
    # Save to CSV with f in the filename
    f_str = str(f).replace('.', '_')  # e.g., 0.3 → '0_3' for filename
    file_name = f"P_omega_vs_amax_f_{f_str}_CustomBounds.csv"
    df.to_csv(P_omega_data_folder_path + file_name, index=False)
    
    print(f"Saved {file_name}")

Now working on f = 1
Please cite Warren & Brandt (2008) when using these optical constants
Please cite Draine 2003 when using these optical constants
Reading opacities from troilitek
Please cite Henning & Stognienko (1996) when using these optical constants
Reading opacities from organicsk
Please cite Henning & Stognienko (1996) when using these optical constants
| material                            | volume fractions | mass fractions |
|-------------------------------------|------------------|----------------|
| Water Ice (Warren & Brandt 2008)    | 0.3642           | 0.2            |
| Astronomical Silicates (Draine 2003)| 0.167            | 0.329          |
| Troilite (Henning)                  | 0.02578          | 0.07434        |
| Organics (Henning)                  | 0.443            | 0.3966         |
Mie ... Done!
Mie ... Done!
Saved P_omega_vs_amax_f_1_CustomBounds.csv
Now working on f = 0.75
Please cite Warren & Brandt (2008) when using these optical constants
Please cite D

/opt/anaconda3/lib/python3.9/site-packages/dsharp_opac/dsharp_opac.py:2116: UserWarning: Maximum error of 1.3%: above error tolerance
  warnings.warn(


Mie ... Done!
Mie ... Done!
Saved P_omega_vs_amax_f_0_25_CustomBounds.csv
Now working on f = 0.3
Please cite Warren & Brandt (2008) when using these optical constants
Please cite Draine 2003 when using these optical constants
Reading opacities from troilitek
Please cite Henning & Stognienko (1996) when using these optical constants
Reading opacities from organicsk
Please cite Henning & Stognienko (1996) when using these optical constants
| material                            | volume fractions | mass fractions |
|-------------------------------------|------------------|----------------|
| Water Ice (Warren & Brandt 2008)    | 0.3642           | 0.2            |
| Astronomical Silicates (Draine 2003)| 0.167            | 0.329          |
| Troilite (Henning)                  | 0.02578          | 0.07434        |
| Organics (Henning)                  | 0.443            | 0.3966         |
using Maxwell-Garnett mixing: first component should be host material (= matrix)
    matrix = Vacuum
M

/opt/anaconda3/lib/python3.9/site-packages/dsharp_opac/dsharp_opac.py:2116: UserWarning: Maximum error of 20%: above error tolerance
  warnings.warn(


Mie ... Done!
Mie ... Done!
Saved P_omega_vs_amax_f_0_3_CustomBounds.csv
Now working on f = 0.1
Please cite Warren & Brandt (2008) when using these optical constants
Please cite Draine 2003 when using these optical constants
Reading opacities from troilitek
Please cite Henning & Stognienko (1996) when using these optical constants
Reading opacities from organicsk
Please cite Henning & Stognienko (1996) when using these optical constants
| material                            | volume fractions | mass fractions |
|-------------------------------------|------------------|----------------|
| Water Ice (Warren & Brandt 2008)    | 0.3642           | 0.2            |
| Astronomical Silicates (Draine 2003)| 0.167            | 0.329          |
| Troilite (Henning)                  | 0.02578          | 0.07434        |
| Organics (Henning)                  | 0.443            | 0.3966         |
using Maxwell-Garnett mixing: first component should be host material (= matrix)
    matrix = Vacuum
Mi

/opt/anaconda3/lib/python3.9/site-packages/dsharp_opac/dsharp_opac.py:2116: UserWarning: Maximum error of 9.5%: above error tolerance
  warnings.warn(


Mie ... Done!
Mie ... Done!
Saved P_omega_vs_amax_f_0_1_CustomBounds.csv
Now working on f = 0.01
Please cite Warren & Brandt (2008) when using these optical constants
Please cite Draine 2003 when using these optical constants
Reading opacities from troilitek
Please cite Henning & Stognienko (1996) when using these optical constants
Reading opacities from organicsk
Please cite Henning & Stognienko (1996) when using these optical constants
| material                            | volume fractions | mass fractions |
|-------------------------------------|------------------|----------------|
| Water Ice (Warren & Brandt 2008)    | 0.3642           | 0.2            |
| Astronomical Silicates (Draine 2003)| 0.167            | 0.329          |
| Troilite (Henning)                  | 0.02578          | 0.07434        |
| Organics (Henning)                  | 0.443            | 0.3966         |
using Maxwell-Garnett mixing: first component should be host material (= matrix)
    matrix = Vacuum
M

/opt/anaconda3/lib/python3.9/site-packages/dsharp_opac/dsharp_opac.py:2116: UserWarning: Maximum error of 1.4e+03%: above error tolerance
  warnings.warn(


Mie ... Done!
Mie ... Done!
Saved P_omega_vs_amax_f_0_01_CustomBounds.csv


/opt/anaconda3/lib/python3.9/site-packages/dsharp_opac/dsharp_opac.py:2116: UserWarning: Maximum error of 1.6e+07%: above error tolerance
  warnings.warn(
